# 🧠 MedSeg-XAI: Master Heatmap Renderer & MPRT Saliency Audit
This standalone Colab notebook runs `load_case()`, `segment()`, `explain()`, and `audit()` on all 5 benchmark cases (512x512x3), rendering high-resolution heatmaps and cascading degradation plots.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# Auto-detect code directory across Drive and local
for p in [Path('/content/drive/MyDrive/medsegxai/code'), Path('code'), Path('../code')]:
    if p.exists():
        sys.path.insert(0, str(p))
        break

from load_case import load_case
from segment import segment, compute_dice
from explain import explain
from audit import audit
print('Loaded all 4 core functions!')

In [ ]:
fig, axes = plt.subplots(5, 4, figsize=(16, 20), dpi=130)
for i in range(1, 6):
    case_id = f'case_{i:02d}'
    img_3ch, true_mask, box = load_case(case_id)
    img_gray = img_3ch[:, :, 0]
    pred_mask = segment(img_3ch, box)
    dice = compute_dice(pred_mask, true_mask)
    hm = explain(img_3ch, box)
    scores, verdict = audit(img_3ch, box)
    
    # 1. Raw
    axes[i-1, 0].imshow(img_gray, cmap='gray')
    axes[i-1, 0].add_patch(patches.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], edgecolor='#FF3366', facecolor='none', linewidth=1.5))
    axes[i-1, 0].set_title(f'{case_id.upper()} Raw MRI', fontsize=10, fontweight='bold')
    axes[i-1, 0].axis('off')
    
    # 2. Seg
    axes[i-1, 1].imshow(img_gray, cmap='gray')
    overlay = np.zeros((*img_gray.shape, 4))
    overlay[pred_mask > 0] = [0, 0.9, 0.2, 0.45]
    axes[i-1, 1].imshow(overlay)
    axes[i-1, 1].set_title(f'Dice: {dice:.4f}', fontsize=10, fontweight='bold', color='green')
    axes[i-1, 1].axis('off')
    
    # 3. Heatmap
    axes[i-1, 2].imshow(img_gray, cmap='gray')
    axes[i-1, 2].imshow(hm, cmap='inferno', alpha=0.55)
    axes[i-1, 2].set_title('Attention Heatmap', fontsize=10, fontweight='bold', color='purple')
    axes[i-1, 2].axis('off')
    
    # 4. Degradation Curve
    axes[i-1, 3].plot(['Clean', 'Dec', 'Mem', 'Casc'], list(scores.values()), marker='o', color='green')
    axes[i-1, 3].axhline(0.30, color='gray', linestyle='--')
    axes[i-1, 3].set_ylim(-0.05, 1.05)
    axes[i-1, 3].set_title(f'Audit: {verdict}', fontsize=10, fontweight='bold', color='green')
    axes[i-1, 3].grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.show()